# Scout candidate lifecycle: interactive exploration

Walks a single *synthetic* JPL Scout NEO candidate through its lifecycle --
**announced -> updated -> first Rubin ToO pass -> still passing -> retired** --
by replaying the same mocked-Scout-API ingest path used in
`solsys_code/tests/test_scout_views.py::RubinTooScoutIngestEndToEndTest`, but as a
*sequence* of distinct `lastRun` snapshots so each stage appends a row to
`tom_jpl.models.ScoutDetailHistory`.

Goals:
- Sanity-check `solsys_code.rubin_too.evaluate_filters()` / `passes_filters()` against
  a realistic multi-stage history -- exactly the kind of data the still-to-build #3b
  per-year first-pass aggregation view will consume.
- Watch *which* filter(s) flip a candidate from failing to passing, and see how
  `ScoutDetail.active` behaves once Scout stops listing it.

**Isolation:** this notebook spins up a throwaway Django test database (in-memory for
SQLite) via the test runner, so it never touches `src/fomo_db.sqlite3`. It is
deliberately *not* wired into `docs/notebooks.rst` / the Sphinx build -- it's a dev
exploration tool, not rendered documentation.

**Run it** from the FOMO dev environment with the editable `tom_jpl` checkout on the
path (this feature needs `tom_jpl`'s `vmag`/`rate`/`ra`/`dec`/`t_ephem` `ScoutDetail`
fields and `ScoutDetailHistory`, same prerequisite as `test_scout_views.py`):

```
PYTHONPATH=/home/tlister/git/tom_jpl jupyter notebook docs/notebooks/scout_lifecycle_exploration.ipynb
```


In [ ]:
import os
import sys
from pathlib import Path

import django

# Jupyter's IPython kernel runs its own asyncio event loop; Django's synchronous DB
# layer refuses to run inside one unless this is set (it's a notebook-only quirk --
# `manage.py`/`pytest` run outside an event loop and don't need it).
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')


def find_project_root(start=None):
    """Walk upward from `start` (default: cwd) until a directory containing manage.py."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'manage.py').exists():
            return candidate
    raise RuntimeError('Could not locate the FOMO project root (no manage.py found above cwd)')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'src.fomo.settings')
django.setup()

In [ ]:
# Spin up an isolated test database (in-memory for SQLite) -- mirrors what
# `manage.py test` does, so the ORM is fully usable without touching the dev DB.
from django.test.runner import DiscoverRunner
from django.test.utils import setup_test_environment

setup_test_environment()
_runner = DiscoverRunner(verbosity=0)
_old_db_config = _runner.setup_databases()

## Replaying the Scout ingest pipeline

The helpers below are adapted from `RubinTooScoutIngestEndToEndTest` in
`solsys_code/tests/test_scout_views.py`: `requests.get` is patched at the HTTP
boundary so the *real* `ScoutDataService.query_targets` / `to_target` chain runs
end-to-end, landing rows in `ScoutDetail` (current state) and `ScoutDetailHistory`
(append-only, deduped on `(target, last_run)`) exactly as production ingest would.

`run_ingest_cycle` represents one Scout poll where the candidate *is* listed;
`run_departure_cycle` represents one where it has dropped out, mirroring the
`ingest_scout` management command's sweep (`Command._sweep`):
`ScoutDetail.objects.filter(active=True).exclude(target_id__in=seen_ids).update(active=False)`.


In [ ]:
from unittest import mock

from django.contrib.auth.models import AnonymousUser
from django.test import RequestFactory
from tom_jpl.jpl import ScoutDataService
from tom_jpl.models import ScoutDetail, ScoutDetailHistory
from tom_targets.models import Target

from solsys_code.rubin_too import RUBIN_TOO_FILTERS, evaluate_filters, passes_filters

_SCOUT_SIGNATURE = {'source': 'NASA/JPL Scout API', 'version': '1.3'}

# Loose enough that every synthetic snapshot below survives ScoutDataService's own
# input-parameter pre-filter and reaches `to_target`.
_PERMISSIVE_INPUT_PARAMETERS = {
    'ca_dist_min': None,
    'data_service': 'Scout',
    'geo_score_max': 5,
    'impact_rating_min': None,
    'neo_score_min': None,
    'pha_score_min': None,
    'pos_unc_max': None,
    'pos_unc_min': None,
    'query_name': '',
    'query_save': False,
    'tdes': '',
}

# A minimal single-solution orbit block; H is fixed at 22.0 throughout (Target.abs_mag
# does not drift in this synthetic run -- see the note on H below).
_SCOUT_ORBITS = {
    'fields': [
        'idx',
        'epoch',
        'ec',
        'qr',
        'tp',
        'om',
        'w',
        'inc',
        'H',
        'dca',
        'tca',
        'moid',
        'vinf',
        'geoEcc',
        'impFlag',
    ],
    'data': [
        [
            0,
            '2461079.712931752',
            '8.855752093403347E-01',
            '4.998861947435592E-01',
            '2461038.529885748',
            '1.3898404962804094E+02',
            '2.6665272638172337E+02',
            '1.4692644033445657E+01',
            '22.0',
            '4.34144690247134E-03',
            '2.4610794091831E+06',
            '1.971147172E-03',
            '2.790457665E+01',
            '1.278753791E+03',
            0,
        ]
    ],
    'count': '1',
}


def _scout_api_get(summary_object, detail_object):
    """`requests.get` replacement answering the Scout summary-list and per-object calls."""
    summary_payload = {'signature': _SCOUT_SIGNATURE, 'count': 1, 'data': [summary_object]}
    detail_payload = dict(detail_object, signature=_SCOUT_SIGNATURE)

    def _get(url, data=None, **kwargs):
        payload = detail_payload if (data and data.get('tdes')) else summary_payload
        response = mock.Mock()
        response.json.return_value = payload
        return response

    return _get


def run_ingest_cycle(scout_object):
    """Run one Scout poll where `scout_object` is (still) listed.

    Returns the resulting `ScoutDetail` so the caller can inspect/evaluate it.
    """
    ds = ScoutDataService()
    detail_object = dict(scout_object, orbits=_SCOUT_ORBITS)
    fake_get = _scout_api_get(scout_object, detail_object)

    request = RequestFactory().get('/')
    request.user = AnonymousUser()
    with mock.patch('tom_jpl.jpl.requests.get', side_effect=fake_get):
        targets_data = ds.query_targets(dict(_PERMISSIVE_INPUT_PARAMETERS))
    with mock.patch('tom_dataservices.dataservices.messages'):
        for target_data in targets_data:
            ds.to_target(target_data, request=request)

    return ScoutDetail.objects.get(target__name=scout_object['objectName'])


def run_departure_cycle():
    """Run one Scout poll where the candidate is no longer listed at all.

    Mirrors `ingest_scout`'s sweep. We have only one candidate in this isolated
    DB, so 'not seen this cycle' is simply 'everything still flagged active'.
    History rows are untouched -- only `ScoutDetail.active` changes.
    """
    ScoutDetail.objects.filter(active=True).update(active=False)

## The lifecycle, staged

Four synthetic Scout snapshots for the same fictitious object (`ZTF26AA`, a
Northern -- `dec > 0` -- candidate, so the N thresholds for `Vmag`/`uncP1` apply),
each with its own `lastRun` so every stage appends one `ScoutDetailHistory` row.
Watch `neoScore`, `geocentricScore`, `rating`, `rmsN`, `nObs`/`arc`, `Vmag`, `uncP1`
and `rate` climb from "nothing special" to "passes every Section 2.1 filter at once".


In [ ]:
LIFECYCLE_STAGES = [
    dict(
        label='Announced',
        narrative='First Scout listing: a handful of observations, a short arc, nothing stands out yet.',
        objectName='ZTF26AA',
        neoScore=80,
        neo1kmScore=0,
        phaScore=0,
        ieoScore=0,
        geocentricScore=4,
        rating=1,
        unc='18',
        uncP1='25',
        caDist='1.4',
        arc='0.5',
        nObs=3,
        rmsN='1.4',
        Vmag='20.2',
        rate='14.0',
        ra='10:30',
        dec='+12',
        tEphem='2026-01-05 06:00',
        lastRun='2026-01-05 10:00',
    ),
    dict(
        label='Early follow-up',
        narrative='More observations and a longer arc tighten the orbit fit; impact metrics are still modest.',
        objectName='ZTF26AA',
        neoScore=93,
        neo1kmScore=0,
        phaScore=0,
        ieoScore=0,
        geocentricScore=2,
        rating=2,
        unc='30',
        uncP1='40',
        caDist='1.1',
        arc='4.0',
        nObs=8,
        rmsN='0.7',
        Vmag='20.9',
        rate='11.0',
        ra='10:32',
        dec='+12',
        tEphem='2026-01-09 08:00',
        lastRun='2026-01-09 11:30',
    ),
    dict(
        label='First Rubin ToO pass',
        narrative='Every Section 2.1 filter is satisfied at once -- the operational ToO-trigger moment.',
        objectName='ZTF26AA',
        neoScore=99,
        neo1kmScore=0,
        phaScore=0,
        ieoScore=0,
        geocentricScore=1,
        rating=4,
        unc='55',
        uncP1='80',
        caDist='0.6',
        arc='9.0',
        nObs=14,
        rmsN='0.3',
        Vmag='21.9',
        rate='7.0',
        ra='10:35',
        dec='+12',
        tEphem='2026-01-14 04:00',
        lastRun='2026-01-14 09:15',
    ),
    dict(
        label='Still passing',
        narrative='The orbit keeps refining; the candidate remains inside the Rubin ToO window.',
        objectName='ZTF26AA',
        neoScore=99,
        neo1kmScore=0,
        phaScore=0,
        ieoScore=0,
        geocentricScore=0,
        rating=4,
        unc='60',
        uncP1='95',
        caDist='0.5',
        arc='16.0',
        nObs=22,
        rmsN='0.2',
        Vmag='22.1',
        rate='6.0',
        ra='10:39',
        dec='+12',
        tEphem='2026-01-21 10:00',
        lastRun='2026-01-21 14:00',
    ),
]

seen_first_pass = False
for stage in LIFECYCLE_STAGES:
    scout_object = {k: v for k, v in stage.items() if k not in ('label', 'narrative')}
    scout_detail = run_ingest_cycle(scout_object)
    verdict = evaluate_filters(scout_detail)
    failing = [label for key, label, _f in RUBIN_TOO_FILTERS if not verdict[key]]
    passes = passes_filters(scout_detail)

    print(f"=== {stage['label']}  (lastRun {stage['lastRun']}) ===")
    print(stage['narrative'])
    print(f'  passes_filters() = {passes}    ScoutDetail.active = {scout_detail.active}')
    if failing:
        print('  failing filters:')
        for label in failing:
            print(f'    - {label}')
    elif not seen_first_pass:
        print('  failing filters: none -- this is the first-pass event')
    else:
        print('  failing filters: none -- still inside the passing window')
    print()

    seen_first_pass = seen_first_pass or passes

## What `ScoutDetailHistory` now holds (the #3b data source)

Each cycle above appended exactly one row (deduped on `(target, last_run)`), giving
the temporal trail #3b's per-year first-pass aggregation needs to walk.
`evaluate_filters()` / `passes_filters()` work unchanged on history rows -- they only
touch the fields shared via `BaseScoutDetail`.

> **Heads-up for #3b:** `ScoutDetailHistory` doesn't snapshot `H` itself (it lives on
> `Target.abs_mag`, which can drift as the orbit is refined). `passes_filters(row)`
> below falls back to the *current* `Target.abs_mag` -- harmless here since H is
> static for this synthetic run, but the real per-year aggregation should pass the
> contemporaneous `abs_mag` explicitly (`evaluate_filters(row, abs_mag=...)`) per the
> design note: snapshot it at ingest time rather than relying on the live value.


In [ ]:
target = Target.objects.get(name='ZTF26AA')
history = ScoutDetailHistory.objects.filter(target=target).order_by('last_run')
first_pass = next((row for row in history if passes_filters(row)), None)

print(f'{history.count()} ScoutDetailHistory rows for {target.name}:')
for row in history:
    marker = '  <-- first-pass event' if row.pk == getattr(first_pass, 'pk', None) else ''
    print(f'  {row.last_run}   passes_filters={passes_filters(row)}{marker}')

## Retirement: the candidate drops off Scout

Eventually Scout stops listing the object (ruled out, reclassified, or just aged off
the candidate list). The next poll sees an empty/unrelated summary, so the
`ingest_scout` sweep marks it `active=False` -- the *history* is left untouched.


In [ ]:
print(f'Before retirement: ScoutDetail.active = {ScoutDetail.objects.get(target=target).active}')

run_departure_cycle()

retired = ScoutDetail.objects.get(target=target)
print(f'After a Scout poll where it is no longer listed: active = {retired.active}')
print(f'ScoutDetailHistory rows preserved: {ScoutDetailHistory.objects.filter(target=target).count()}')

## (Optional) Check the live `/scout/rubin-too/` view post-retirement

Hitting the view resolves the URL conf, which imports `solsys_code.views` -- the
module whose load triggers `fomo_furnish_spiceypy()` and (per `CLAUDE.md`) a ~1.6 GB
SPICE-kernel download to `~/.cache/sorcha/` on a fresh machine. If you've already run
the ephemeris features or `manage.py test`, the kernels are cached and this just pays
a slower import. Skip this cell if you'd rather not trigger it.


In [ ]:
from django.test import Client
from django.urls import reverse

client = Client()
response = client.get(reverse('scout_rubin_too'))
listed = target.name.encode() in response.content

print(f'ScoutDetail.active = {ScoutDetail.objects.get(target=target).active}')
print(f"'{target.name}' still appears in the live 'currently passing' view: {listed}")
print(f"num_passing = {response.context['num_passing']}, num_total = {response.context['num_total']}")

### Finding: retirement does not remove a candidate from the live view

`RubinTooScoutListView.get_queryset()` (`solsys_code/scout_views.py`) filters
`ScoutDetail.objects.select_related('target').all()` purely on `passes_filters(sd)` --
it never checks `active`. So once Scout drops a candidate (sweep flips
`active=False`), it keeps showing up under "currently passing" for as long as its
*last-known* snapshot numerically satisfies the filters, which for an object retired
while still passing is indefinitely.

That looks like a latent gap relative to the Section 2.3 intent ("cancel once a
filter stops being true", reasonably extended to "object no longer tracked by
Scout at all"). Adding `.filter(active=True)` to the queryset would likely close it --
worth raising as a follow-up regardless of how #3b proceeds.


## Teardown

Destroy the throwaway test database and restore the test environment.

In [ ]:
from django.test.utils import teardown_test_environment

_runner.teardown_databases(_old_db_config)
teardown_test_environment()